# Phase 3: Real-Time Face Mask Detection

In this final notebook, I'm going to merge my trained PyTorch model with OpenCV. I will use OpenCV to access my webcam and detect where faces are in the video feed. Then, I'll extract those face regions, pass them to my MobileNetV2 model, and draw a bounding box with the predicted label (Mask or No Mask) right on the screen!

In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms, models
import cv2
import numpy as np
from PIL import Image

# Setting up the device again
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Loading the Trained Model

I need to recreate the exact same model architecture I used in Phase 2, and then load the weights that I saved into it. I also need to make sure I put the model into evaluation mode (`model.eval()`) since I am not training it anymore.

In [ ]:
# 1. Recreate the MobileNetV2 architecture
model = models.mobilenet_v2()
num_ftrs = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_ftrs, 2) 

# 2. Load the saved weights from Phase 2
model_path = '../saved_models/face_mask_mobilenetv2.pth'
model.load_state_dict(torch.load(model_path, map_location=device))

# 3. Move to device and set to evaluation mode
model = model.to(device)
model.eval()

print("Trained model successfully loaded and ready for inference!")

## Preparing the Image Transformations and Face Detector

I have to apply the exact same transformations to my webcam frames as I did to my training images. For finding faces quickly, I'll use OpenCV's built-in Haar Cascade classifier, which is lightweight and perfect for CPU real-time processing.

In [ ]:
# The exact same transformations from Phase 1 and 2
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Load OpenCV's pre-trained face detection algorithm
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

# Define my classes and colors for the bounding boxes (BGR format for OpenCV)
classes = ['With Mask', 'Without Mask']
colors = [(0, 255, 0), (0, 0, 255)] # Green for Mask, Red for No Mask

## The Real-Time Webcam Loop

This is the main event! I will open the webcam stream, read it frame by frame, detect faces, predict the mask status, and display the results. 

**Note to self: Press 'q' on the keyboard to exit the video window.**

In [ ]:
# Start the webcam (0 is usually the default built-in camera)
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Error: Could not open webcam.")
else:
    print("Webcam opened. Press 'q' in the video window to quit.")

while cap.isOpened():
    # Read a single frame from the camera
    ret, frame = cap.read()
    if not ret:
        break
        
    # OpenCV uses BGR, but Haar Cascades work best on grayscale
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    # Detect faces in the grayscale frame
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(60, 60))
    
    for (x, y, w, h) in faces:
        # Extract the face region of interest (ROI)
        face_roi = frame[y:y+h, x:x+w]
        
        # Convert the OpenCV BGR image to a PIL Image (RGB) so PyTorch can use it
        face_pil = Image.fromarray(cv2.cvtColor(face_roi, cv2.COLOR_BGR2RGB))
        
        # Apply transformations and add a batch dimension
        input_tensor = preprocess(face_pil).unsqueeze(0).to(device)
        
        # Make the prediction
        with torch.no_grad():
            outputs = model(input_tensor)
            
            # Apply softmax to get percentages (probabilities)
            probabilities = torch.nn.functional.softmax(outputs, dim=1)[0]
            confidence, predicted_class = torch.max(probabilities, 0)
            
            class_idx = predicted_class.item()
            conf_percent = confidence.item() * 100
            
        # Draw the bounding box and label
        label = f"{classes[class_idx]}: {conf_percent:.1f}%"
        color = colors[class_idx]
        
        cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)
        cv2.putText(frame, label, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)
        
    # Display the resulting frame
    cv2.imshow('Face Mask Detector - Press Q to Exit', frame)
    
    # Break the loop if the 'q' key is pressed
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release the webcam and close all windows
cap.release()
cv2.destroyAllWindows()